In [1]:
import gc
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from concurrent.futures import ThreadPoolExecutor
from math import ceil
from pathlib import Path
from time import perf_counter

import h5py
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForImageTextToText, AutoProcessor

/home/tdnguyen/miniforge3/envs/cxr-vlm-interp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set paths and extraction settings.

DATA_CSV = Path("artifacts/processed_data/chexpertplus_frontal_5labels.csv")
OUT_H5 = Path("artifacts/embeddings/experiment_a_image_only_embeddings.h5")

MEDSIGLIP_MODEL_ID = "google/medsiglip-448"
MEDGEMMA_MODEL_ID = "google/medgemma-4b-it"

CACHE_NUM_WORKERS = 4
MEDSIGLIP_BATCH_SIZE = 512
MEDGEMMA_BATCH_SIZE = 24

MODEL_DTYPE = torch.bfloat16
SAVE_DTYPE = np.float32

TARGET_LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]

print(DATA_CSV, DATA_CSV.exists())
print(OUT_H5)

artifacts/processed_data/chexpertplus_frontal_5labels.csv True
artifacts/embeddings/experiment_a_image_only_embeddings.h5


In [3]:
# Load the processed data.

df = pd.read_csv(DATA_CSV)

print("rows", len(df))
display(df["probe_split"].value_counts().to_frame("rows"))
display(df.head())

rows 25000


,rows
probe_split,
train,20000
test,5000


,subject_id,study_id,dicom_id,split,probe_split,ViewPosition,view,image_path,Atelectasis,Cardiomegaly,Consolidation,Edema,Pleural Effusion
0,patient64544,patient64544_study1,patient64544_study1_view1_frontal,test,train,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0
1,patient64548,patient64548_study1,patient64548_study1_view1_frontal,test,test,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0
2,patient64555,patient64555_study1,patient64555_study1_view1_frontal,test,test,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0
3,patient64565,patient64565_study1,patient64565_study1_view1_frontal,test,train,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,1,0,0,0
4,patient64578,patient64578_study1,patient64578_study1_view1_frontal,test,train,AP,AP,/opt/gpudata/cxr/chexpertplus/PNG/valid/patien...,0,0,0,0,0


In [4]:
# Helper functions.

def load_images(paths):
    return [Image.open(path).convert("RGB") for path in paths]


def move_inputs(inputs, device, dtype):
    moved = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            if value.is_floating_point():
                moved[key] = value.to(device=device, dtype=dtype)
            else:
                moved[key] = value.to(device=device)
        else:
            moved[key] = value
    return moved


def batches(n, batch_size):
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        yield start, end


def open_rgb(path):
    return Image.open(path).convert("RGB")


def cache_images(paths, num_workers):
    if num_workers == 1:
        return [open_rgb(path) for path in tqdm(paths, desc="Caching images")]
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        return list(tqdm(pool.map(open_rgb, paths), total=len(paths), desc=f"Caching images workers={num_workers}"))


def cpu_batches(items, batch_size):
    for start in range(0, len(items), batch_size):
        end = min(start + batch_size, len(items))
        yield start, end, items[start:end]


def cache_inputs_on_cpu(inputs):
    cached = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            if value.is_floating_point():
                cached[key] = value.to(device="cpu", dtype=MODEL_DTYPE).contiguous()
            else:
                cached[key] = value.to(device="cpu").contiguous()
        else:
            cached[key] = value
    return cached


class ImagePathDataset(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        return index, image


def image_collate(batch):
    indices, images = zip(*batch)
    return torch.tensor(indices), list(images)


def make_image_loader(paths, batch_size, num_workers):
    return DataLoader(
        ImagePathDataset(paths),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=image_collate,
        pin_memory=True,
        persistent_workers=num_workers > 0,
        prefetch_factor=4 if num_workers > 0 else None,
    )

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


def reset_cuda_memory_stats():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


In [5]:
# Combined benchmark using image caching only.

BENCHMARK_ROWS = 2048
BENCHMARK_H5 = Path("artifacts/embeddings/combined_image_cache_benchmark_embeddings.h5")

device = "cuda" if torch.cuda.is_available() else "cpu"
benchmark_df = df.iloc[:BENCHMARK_ROWS].reset_index(drop=True)
benchmark_paths = benchmark_df["image_path"].tolist()

if BENCHMARK_H5.exists():
    BENCHMARK_H5.unlink()
BENCHMARK_H5.parent.mkdir(parents=True, exist_ok=True)

benchmark_times = {}

# Cache images once in CPU RAM.
t0 = perf_counter()
cached_images = cache_images(benchmark_paths, CACHE_NUM_WORKERS)
benchmark_times["image_cache_sec"] = perf_counter() - t0

# Create temporary HDF5 output.
str_dtype = h5py.string_dtype(encoding="utf-8")
with h5py.File(BENCHMARK_H5, "w") as h5:
    h5.create_dataset("study_id", data=benchmark_df["study_id"].astype(str).to_numpy(), dtype=str_dtype)
    h5.create_dataset("labels", data=benchmark_df[TARGET_LABELS].to_numpy(dtype=np.int8))
    h5.create_group("medsiglip")
    h5.create_group("medgemma")

# MedSigLIP processor + model + HDF5 write.
medsiglip_processor = AutoProcessor.from_pretrained(MEDSIGLIP_MODEL_ID)
medsiglip_model = AutoModel.from_pretrained(MEDSIGLIP_MODEL_ID, dtype=MODEL_DTYPE).to(device).eval()
medsiglip_dim = medsiglip_model.config.vision_config.hidden_size

medsiglip_processor_time = 0.0
medsiglip_move_time = 0.0
medsiglip_forward_time = 0.0
medsiglip_cpu_time = 0.0
medsiglip_write_time = 0.0
medsiglip_flush_time = 0.0

with h5py.File(BENCHMARK_H5, "a") as h5:
    dset = h5.create_dataset("medsiglip/global", shape=(len(benchmark_df), medsiglip_dim), dtype="float32")

    for start, end, images in tqdm(cpu_batches(cached_images, MEDSIGLIP_BATCH_SIZE), total=ceil(len(cached_images) / MEDSIGLIP_BATCH_SIZE), desc="MedSigLIP image-cache benchmark"):
        t0 = perf_counter()
        inputs = medsiglip_processor(images=images, return_tensors="pt")
        medsiglip_processor_time += perf_counter() - t0

        t0 = perf_counter()
        inputs = move_inputs(inputs, device, MODEL_DTYPE)
        if device == "cuda":
            torch.cuda.synchronize()
        medsiglip_move_time += perf_counter() - t0

        t0 = perf_counter()
        with torch.inference_mode():
            outputs = medsiglip_model.get_image_features(**inputs)
            emb = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
            emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
        if device == "cuda":
            torch.cuda.synchronize()
        medsiglip_forward_time += perf_counter() - t0

        t0 = perf_counter()
        emb_np = emb.detach().cpu().float().numpy().astype(SAVE_DTYPE)
        medsiglip_cpu_time += perf_counter() - t0

        t0 = perf_counter()
        dset[start:end] = emb_np
        medsiglip_write_time += perf_counter() - t0

        del inputs, outputs, emb, emb_np

    t0 = perf_counter()
    h5.flush()
    medsiglip_flush_time += perf_counter() - t0

benchmark_times["medsiglip_processor_sec"] = medsiglip_processor_time
benchmark_times["medsiglip_move_to_gpu_sec"] = medsiglip_move_time
benchmark_times["medsiglip_forward_sec"] = medsiglip_forward_time
benchmark_times["medsiglip_cpu_copy_sec"] = medsiglip_cpu_time
benchmark_times["medsiglip_hdf5_write_sec"] = medsiglip_write_time
benchmark_times["medsiglip_hdf5_flush_sec"] = medsiglip_flush_time

del medsiglip_model, medsiglip_processor
cleanup_memory()

# MedGemma processor + model + feature extraction + HDF5 write.
medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)
medgemma_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_MODEL_ID,
    dtype=MODEL_DTYPE,
    device_map="auto",
).eval()

model_device = next(medgemma_model.parameters()).device
num_layers = medgemma_model.config.text_config.num_hidden_layers
hidden_size = medgemma_model.config.text_config.hidden_size
image_token_id = getattr(medgemma_model.config, "image_token_id", None)
if image_token_id is None:
    image_token_id = getattr(medgemma_model.config, "image_token_index")

medgemma_processor_time = 0.0
medgemma_move_time = 0.0
medgemma_forward_time = 0.0
medgemma_feature_time = 0.0
medgemma_write_time = 0.0
medgemma_flush_time = 0.0

reset_cuda_memory_stats()

with h5py.File(BENCHMARK_H5, "a") as h5:
    projected_dset = h5.create_dataset("medgemma/projected_image_mean", shape=(len(benchmark_df), hidden_size), dtype="float32")
    mean_dset = h5.create_dataset("medgemma/layer_image_mean", shape=(len(benchmark_df), num_layers, hidden_size), dtype="float32")
    last_dset = h5.create_dataset("medgemma/layer_last_image", shape=(len(benchmark_df), num_layers, hidden_size), dtype="float32")

    for start, end, images in tqdm(cpu_batches(cached_images, MEDGEMMA_BATCH_SIZE), total=ceil(len(cached_images) / MEDGEMMA_BATCH_SIZE), desc="MedGemma image-cache benchmark"):
        t0 = perf_counter()
        messages = [[{"role": "user", "content": [{"type": "image", "image": image}]}] for image in images]
        inputs = medgemma_processor.apply_chat_template(
            messages,
            add_generation_prompt=False,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            processor_kwargs={"padding": True},
        )
        medgemma_processor_time += perf_counter() - t0

        t0 = perf_counter()
        inputs = move_inputs(inputs, model_device, MODEL_DTYPE)
        if device == "cuda":
            torch.cuda.synchronize()
        medgemma_move_time += perf_counter() - t0

        t0 = perf_counter()
        with torch.inference_mode():
            outputs = medgemma_model(
                **inputs,
                output_hidden_states=True,
                use_cache=False,
                logits_to_keep=1,
                return_dict=True,
            )
        if device == "cuda":
            torch.cuda.synchronize()
        medgemma_forward_time += perf_counter() - t0

        t0 = perf_counter()
        image_mask = inputs["input_ids"].eq(image_token_id)
        batch_size = end - start
        image_token_counts = image_mask.sum(dim=1)
        image_token_count = int(image_token_counts[0].item())
        if not bool(torch.all(image_token_counts == image_token_count).item()):
            raise ValueError("Different image token counts inside the same batch")

        projected_np = outputs.image_hidden_states.float().mean(dim=1).detach().cpu().numpy().astype(SAVE_DTYPE)
        layer_image_mean = torch.empty((batch_size, num_layers, hidden_size), dtype=torch.float32)
        layer_last_image = torch.empty((batch_size, num_layers, hidden_size), dtype=torch.float32)

        for layer_i, hidden in enumerate(outputs.hidden_states[1:]):
            image_hidden = hidden[image_mask].reshape(batch_size, image_token_count, hidden_size).float()
            layer_image_mean[:, layer_i] = image_hidden.mean(dim=1).detach().cpu()
            layer_last_image[:, layer_i] = image_hidden[:, -1].detach().cpu()

        layer_mean_np = layer_image_mean.numpy().astype(SAVE_DTYPE)
        layer_last_np = layer_last_image.numpy().astype(SAVE_DTYPE)
        medgemma_feature_time += perf_counter() - t0

        t0 = perf_counter()
        projected_dset[start:end] = projected_np
        mean_dset[start:end] = layer_mean_np
        last_dset[start:end] = layer_last_np
        medgemma_write_time += perf_counter() - t0

        del outputs, inputs, projected_np, layer_image_mean, layer_last_image, layer_mean_np, layer_last_np

    t0 = perf_counter()
    h5.flush()
    medgemma_flush_time += perf_counter() - t0

benchmark_times["medgemma_processor_sec"] = medgemma_processor_time
benchmark_times["medgemma_move_to_gpu_sec"] = medgemma_move_time
benchmark_times["medgemma_forward_sec"] = medgemma_forward_time
benchmark_times["medgemma_feature_extract_sec"] = medgemma_feature_time
benchmark_times["medgemma_hdf5_write_sec"] = medgemma_write_time
benchmark_times["medgemma_hdf5_flush_sec"] = medgemma_flush_time

if device == "cuda":
    benchmark_times["torch_peak_allocated_gb"] = torch.cuda.max_memory_allocated() / 1024**3

benchmark_times["benchmark_rows"] = len(benchmark_df)
benchmark_times["benchmark_file_size_gb"] = BENCHMARK_H5.stat().st_size / 1024**3
benchmark_times["total_sec"] = sum(value for key, value in benchmark_times.items() if key.endswith("_sec"))
benchmark_times["estimated_full_25000_rows_hours"] = benchmark_times["total_sec"] * (25000 / len(benchmark_df)) / 3600

benchmark_results = pd.DataFrame([benchmark_times]).T.rename(columns={0: "value"})
display(benchmark_results)

print(f"total_sec: {benchmark_times['total_sec']:.2f}")
print(f"estimated_full_25000_rows_hours: {benchmark_times['estimated_full_25000_rows_hours']:.2f}")

BENCHMARK_H5.unlink()
del medgemma_model, medgemma_processor, cached_images
cleanup_memory()


MedGemma image-cache benchmark: 100%|██████████| 86/86 [02:06<00:00,  1.47s/it]


,value
image_cache_sec,33.560353
medsiglip_processor_sec,98.341278
medsiglip_move_to_gpu_sec,0.764786
medsiglip_forward_sec,6.934260
medsiglip_cpu_copy_sec,0.011813
medsiglip_hdf5_write_sec,0.009736
medsiglip_hdf5_flush_sec,0.000245
medgemma_processor_sec,63.615375
medgemma_move_to_gpu_sec,2.156453
medgemma_forward_sec,55.625411


total_sec: 265.55
estimated_full_25000_rows_hours: 0.90


In [6]:
# Benchmark processor workers with images already cached in RAM.

WORKER_BENCHMARK_ROWS = 2048
PROCESSOR_WORKER_COUNTS = [1, 2, 4, 8, 16]

worker_df = df.iloc[:WORKER_BENCHMARK_ROWS].reset_index(drop=True)
worker_paths = worker_df["image_path"].tolist()

# Keep image caching fixed so this only measures processor parallelism.
worker_images = cache_images(worker_paths, CACHE_NUM_WORKERS)

medsiglip_processor = AutoProcessor.from_pretrained(MEDSIGLIP_MODEL_ID)
medgemma_processor = AutoProcessor.from_pretrained(MEDGEMMA_MODEL_ID)


def process_medsiglip_batch(images):
    inputs = medsiglip_processor(images=images, return_tensors="pt")
    return cache_inputs_on_cpu(inputs)


def process_medgemma_batch(images):
    messages = [[{"role": "user", "content": [{"type": "image", "image": image}]}] for image in images]
    inputs = medgemma_processor.apply_chat_template(
        messages,
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"padding": True},
    )
    return cache_inputs_on_cpu(inputs)


def benchmark_processor_workers(name, images, batch_size, process_fn, worker_counts):
    rows = []
    image_batches = [batch_images for _, _, batch_images in cpu_batches(images, batch_size)]
    for num_workers in worker_counts:
        t0 = perf_counter()
        if num_workers == 1:
            for batch_images in tqdm(image_batches, desc=f"{name} processor workers=1"):
                inputs = process_fn(batch_images)
                del inputs
        else:
            with ThreadPoolExecutor(max_workers=num_workers) as pool:
                futures = [pool.submit(process_fn, batch_images) for batch_images in image_batches]
                for future in tqdm(futures, desc=f"{name} processor workers={num_workers}"):
                    inputs = future.result()
                    del inputs
        elapsed = perf_counter() - t0
        rows.append({
            "model": name,
            "processor_workers": num_workers,
            "batch_size": batch_size,
            "num_batches": len(image_batches),
            "processor_time_sec": elapsed,
            "images_per_sec": len(images) / elapsed,
        })
        cleanup_memory()
    return rows


processor_rows = []
processor_rows.extend(benchmark_processor_workers(
    "medsiglip",
    worker_images,
    MEDSIGLIP_BATCH_SIZE,
    process_medsiglip_batch,
    PROCESSOR_WORKER_COUNTS,
))
processor_rows.extend(benchmark_processor_workers(
    "medgemma",
    worker_images,
    MEDGEMMA_BATCH_SIZE,
    process_medgemma_batch,
    PROCESSOR_WORKER_COUNTS,
))

processor_worker_df = pd.DataFrame(processor_rows)
display(processor_worker_df)

del worker_images, medsiglip_processor, medgemma_processor
cleanup_memory()


medgemma processor workers=16: 100%|██████████| 86/86 [02:07<00:00,  1.48s/it]


,model,processor_workers,batch_size,num_batches,processor_time_sec,images_per_sec
0,medsiglip,1,512,4,54.404293,37.644088
1,medsiglip,2,512,4,58.022053,35.296924
2,medsiglip,4,512,4,88.402858,23.166672
3,medsiglip,8,512,4,66.687405,30.710447
4,medsiglip,16,512,4,81.413606,25.155500
5,medgemma,1,24,86,59.913047,34.182872
6,medgemma,2,24,86,97.832243,20.933794
7,medgemma,4,24,86,112.245250,18.245761
8,medgemma,8,24,86,128.267848,15.966589
9,medgemma,16,24,86,142.503907,14.371536
